In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
!apt-get update -qq
!apt-get install -y -qq poppler-utils tesseract-ocr
!pip install -q pdfplumber pdf2image pytesseract spacy pandas pillow
!python -m spacy download en_core_web_sm

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.7 MB/s et

In [18]:
import os
import re
import pdfplumber
import pytesseract
import pandas as pd
import spacy

from pdf2image import convert_from_path
from PIL import Image

In [19]:
nlp = spacy.load("en_core_web_sm")
print("spaCy loaded successfully")

spaCy loaded successfully


In [20]:
base_folder = "/content/drive/MyDrive/aiassignment3"
print(os.listdir(base_folder))

['student2.ipynb', 'LESO2.pdf', '911-calls-wav2vec2.ipynb']


In [21]:
pdf_file = "LESO2.pdf"
pdf_path = os.path.join(base_folder, pdf_file)

print("PDF path:", pdf_path)
print("Exists:", os.path.exists(pdf_path))

PDF path: /content/drive/MyDrive/aiassignment3/LESO2.pdf
Exists: True


In [33]:
def clean_text(text):
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_pdf_mixed(pdf_path, min_text_threshold=40):
    pages_output = []

    with pdfplumber.open(pdf_path) as pdf:
        scanned_images = convert_from_path(pdf_path, dpi=300)

        for i, page in enumerate(pdf.pages):
            extracted_text = page.extract_text()
            extracted_text = clean_text(extracted_text)

            # If too little text, treat as scanned page and OCR it
            if len(extracted_text) < min_text_threshold:
                img = scanned_images[i]
                ocr_text = pytesseract.image_to_string(img)
                ocr_text = clean_text(ocr_text)

                final_text = ocr_text
                source = "ocr"
            else:
                final_text = extracted_text
                source = "pdf_text"

            pages_output.append({
                "page_number": i + 1,
                "source": source,
                "text": final_text
            })

    return pages_output

In [34]:
pages_data = extract_pdf_mixed(pdf_path)

print("Total pages:", len(pages_data))
for p in pages_data[:5]:
    print("\nPAGE:", p["page_number"], "| SOURCE:", p["source"])
    print(p["text"][:1000])
    print("-" * 80)

Total pages: 75

PAGE: 1 | SOURCE: ocr
May 26, 2015
Sheriff Kelley Cradduck
Major Nathan Atchison Chief Deputy R.L. Conner
 
 
Major Shawn Holloway
 
7 1300 SW 14th Street Bentonville, Arkansas 72712 Phone: 479-271-1011 (Detention) 479-271-1008 (Admin)
RE: Mine Resistant Ambush Protected vehicle acquired through 1033 Program
To Whom It May Concern:
On April 29, 2014, the Benton County Sheriff's Office obtained a Mine Resistant Ambush
Protected vehicle (MRAP) through Law Enforcement Support Office 1033 Program. As of
that date, our office has implemented a policy for the operation and training of the MRAP
vehicle. The purpose of obtaining this vehicle is for it to be utilized by the Benton County
Sheriff's Office Special Weapons and Tactics Team for the protection of officers and
citizens during high risk operations for the safety of persons from threats such as, but not
limited to potential gunfire.
The policy has guidelines for the operation and maintenance of the vehicle. A deputy th

In [35]:
def extract_date(text):
    patterns = [
        r"\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}\b",
        r"\b\d{1,2}/\d{1,2}/\d{2,4}\b",
        r"\b\d{4}-\d{2}-\d{2}\b"
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(0)
    return "Not found"


def extract_department(text):
    dept_patterns = [
        r"\b[A-Z][A-Za-z&.\- ]+ Police Department\b",
        r"\b[A-Z][A-Za-z&.\- ]+ Sheriff's Office\b",
        r"\b[A-Z][A-Za-z&.\- ]+ Sheriff Office\b",
        r"\b[A-Z][A-Za-z&.\- ]+ Police Dept\b"
    ]

    for pattern in dept_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(0).strip()

    doc = nlp(text[:1500])
    orgs = [ent.text for ent in doc.ents if ent.label_ == "ORG"]
    return orgs[0] if orgs else "Not found"


def extract_doc_type(text):
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    top_text = " ".join(lines[:15]).lower()

    if "1033 training proposal" in top_text:
        return "1033 Training Proposal"
    elif "training proposal" in top_text:
        return "Training Proposal"
    elif "standard operating procedure" in top_text:
        return "Standard Operating Procedure"
    elif "operating procedure" in top_text:
        return "Operating Procedure"
    elif "training plan" in top_text:
        return "Training Plan"
    elif "lesson plan" in top_text:
        return "Lesson Plan"
    elif "course outline" in top_text:
        return "Course Outline"
    elif "training request" in top_text:
        return "Training Request"
    elif "mrap operations" in top_text:
        return "MRAP Operations"
    elif "armored rescue vehicle" in top_text:
        return "Armored Rescue Vehicle Operations"
    elif "training" in top_text:
        return "Training Document"
    elif "proposal" in top_text:
        return "Proposal"
    elif "report" in top_text:
        return "Report"
    else:
        return "Not found"


def extract_program(text):
    t = text.lower()

    if "1033" in t or "leso" in t:
        return "1033 Program"
    elif "law enforcement support" in t:
        return "Law Enforcement Support"
    elif "swat" in t or "s.w.a.t" in t:
        return "S.W.A.T Training"
    elif "mrap" in t:
        return "MRAP Training"
    elif "clest" in t or "c.l.e.s.t" in t:
        return "CLEST Continuing Education"
    else:
        return "Not found"


def extract_key_detail(text):
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    keywords = [
        "equipment", "training", "tactical", "gear", "mrap",
        "swat", "vehicle", "1033", "course", "support", "proposal"
    ]

    for line in lines:
        if any(k in line.lower() for k in keywords):
            return line[:200]

    for line in lines:
        if len(line) > 30:
            return line[:200]

    return "Not found"

In [36]:
def is_new_report(page_text, prev_text=None):
    t = page_text.lower()

    strong_signals = [
        "training proposal",
        "training plan",
        "standard operating procedure",
        "incident report",
        "police department",
        "sheriff",
        "law enforcement support",
        "1033 program"
    ]

    hit_count = sum(signal in t for signal in strong_signals)

    # start a new report if the page has strong title-like content
    if hit_count >= 2:
        return True

    # start new report if department changes
    current_dept = extract_department(page_text)
    prev_dept = extract_department(prev_text) if prev_text else None

    if current_dept != "Not found" and prev_dept not in [None, "Not found"] and current_dept != prev_dept:
        return True

    return False

In [37]:
grouped_reports = []
current_report = None

for i, page in enumerate(pages_data):
    text = page["text"]
    prev_text = pages_data[i-1]["text"] if i > 0 else None

    if current_report is None:
        current_report = {
            "start_page": page["page_number"],
            "end_page": page["page_number"],
            "text": text
        }
        continue

    if is_new_report(text, prev_text):
        grouped_reports.append(current_report)
        current_report = {
            "start_page": page["page_number"],
            "end_page": page["page_number"],
            "text": text
        }
    else:
        current_report["end_page"] = page["page_number"]
        current_report["text"] += "\n" + text

if current_report is not None:
    grouped_reports.append(current_report)

print("Grouped reports:", len(grouped_reports))
for r in grouped_reports[:5]:
    print(f"\nPages {r['start_page']} - {r['end_page']}")
    print(r["text"][:600])
    print("-" * 80)

Grouped reports: 61

Pages 1 - 2
May 26, 2015
Sheriff Kelley Cradduck
Major Nathan Atchison Chief Deputy R.L. Conner
 
 
Major Shawn Holloway
 
7 1300 SW 14th Street Bentonville, Arkansas 72712 Phone: 479-271-1011 (Detention) 479-271-1008 (Admin)
RE: Mine Resistant Ambush Protected vehicle acquired through 1033 Program
To Whom It May Concern:
On April 29, 2014, the Benton County Sheriff's Office obtained a Mine Resistant Ambush
Protected vehicle (MRAP) through Law Enforcement Support Office 1033 Program. As of
that date, our office has implemented a policy for the operation and training of the MRAP
vehicle. The purpose of obt
--------------------------------------------------------------------------------

Pages 3 - 3
Benton County Sheriff’s Office
POLICIES AND PROCEDURES
SUBJECT MRAP Operations
Scheduled
Review Date 6/1/16 ISSUE DATE | 6/1/15
APPROVED REVISION
PURPOSE:
The Benton County Sheriff's Office obtained a Mine Resistant Ambush Protected (MRAP)
Vehicle through the LESO 1033 Pr

In [38]:
rows = []

for idx, report in enumerate(grouped_reports):
    text = clean_text(report["text"])

    rows.append({
        "Report_ID": f"RPT_{idx+1:03}",
        "Start_Page": report["start_page"],
        "End_Page": report["end_page"],
        "Department": extract_department(text),
        "Doc_Type": extract_doc_type(text),
        "Date": extract_date(text),
        "Program": extract_program(text),
        "Key_Detail": extract_key_detail(text)
    })

output_df = pd.DataFrame(rows)
output_df

,Report_ID,Start_Page,End_Page,Department,Doc_Type,Date,Program,Key_Detail
0,RPT_001,1,2,the Benton County Sheriff's Office,Training Document,"May 26, 2015",1033 Program,RE: Mine Resistant Ambush Protected vehicle ac...
1,RPT_002,3,3,The Benton County Sheriff's Office,MRAP Operations,6/1/16,1033 Program,SUBJECT MRAP Operations
2,RPT_003,4,4,Benton County Sheriff's Office,Not found,8/1/14,S.W.A.T Training,Parking or storage: when not deployed or at ma...
3,RPT_004,5,6,Benton Police Department,Lesson Plan,4/24/2015,S.W.A.T Training,ARMORED RESCUE VEHICLE (ARV) OPERATIONS
4,RPT_005,7,8,ARV,Training Document,Not found,S.W.A.T Training,training approved by BNPD.
...,...,...,...,...,...,...,...,...
56,RPT_057,71,71,The Rogers Police Department,Training Document,Not found,Law Enforcement Support,"Law Enforcement Support Office ,"
57,RPT_058,72,72,Joint Program Office,Training Document,Not found,MRAP Training,RG-33 MRAP. Vehicle OPNET 1 0 is.
58,RPT_059,73,73,Joint Program Offi,Not found,Not found,Not found,RG-33 M.RA.P. Vehicle OPNET 10 hrs.
59,RPT_060,74,74,The Union County Sheriff's Office,Training Document,"June 1, 2015",1033 Program,Re: Mine Resistant Ambush Protected (MRAP) Veh...


In [42]:
output_df = output_df.drop_duplicates(
    subset=["Department", "Doc_Type", "Date", "Program", "Key_Detail"]
).reset_index(drop=True)

output_df["Report_ID"] = [f"RPT_{i+1:03}" for i in range(len(output_df))]
output_df

,Report_ID,Start_Page,End_Page,Department,Doc_Type,Date,Program,Key_Detail
0,RPT_001,1,2,the Benton County Sheriff's Office,Training Document,"May 26, 2015",1033 Program,RE: Mine Resistant Ambush Protected vehicle ac...
1,RPT_002,3,3,The Benton County Sheriff's Office,MRAP Operations,6/1/16,1033 Program,SUBJECT MRAP Operations
2,RPT_003,4,4,Benton County Sheriff's Office,Not found,8/1/14,S.W.A.T Training,Parking or storage: when not deployed or at ma...
3,RPT_004,5,6,Benton Police Department,Lesson Plan,4/24/2015,S.W.A.T Training,ARMORED RESCUE VEHICLE (ARV) OPERATIONS
4,RPT_005,7,8,ARV,Training Document,Not found,S.W.A.T Training,training approved by BNPD.
...,...,...,...,...,...,...,...,...
56,RPT_057,71,71,The Rogers Police Department,Training Document,Not found,Law Enforcement Support,"Law Enforcement Support Office ,"
57,RPT_058,72,72,Joint Program Office,Training Document,Not found,MRAP Training,RG-33 MRAP. Vehicle OPNET 1 0 is.
58,RPT_059,73,73,Joint Program Offi,Not found,Not found,Not found,RG-33 M.RA.P. Vehicle OPNET 10 hrs.
59,RPT_060,74,74,The Union County Sheriff's Office,Training Document,"June 1, 2015",1033 Program,Re: Mine Resistant Ambush Protected (MRAP) Veh...


In [44]:
csv_path = os.path.join(base_folder, "student2_mixed_pdf_analysis.csv")
output_df.to_csv(csv_path, index=False)
print("Saved CSV to:", csv_path)

Saved CSV to: /content/drive/MyDrive/aiassignment3/student2_mixed_pdf_analysis.csv
